In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
    'torch>=2.3.0', 'torchaudio>=2.3.0',
    'accelerate>=0.30.0', 'transformers>=4.40.0',
    'datasets>=3.0.0', 'soundfile>=0.12.1',
], check=True)

In [ ]:
import os, json, time, math
from pathlib import Path
from datetime import datetime
import yaml, requests
import torch
import numpy as np

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.xla_multiprocessing as xmp
    XLA_AVAILABLE = True
except ImportError:
    XLA_AVAILABLE = False

from accelerate import Accelerator
from huggingface_hub import HfApi
from datasets import load_dataset, Audio
from shared.secrets import load_secrets
from shared.workflow_kernel import WorkflowKernel
from shared.audio_utils import BUCKET_SIZES, quantize_to_bucket

WORK_DIR = Path('/kaggle/working')
CHECKPOINT_DIR = WORK_DIR / 'mms_tts_checkpoint'
CONFIG_DIR = Path('/kaggle/input/S2S-pipline-v2-0-2/config')
MODEL_NAME = 'facebook/mms-tts-urd-script_arabic'
NUM_TRAIN_STEPS = 30000
WARMUP_STEPS = 2000
LEARNING_RATE = 1e-4
BATCH_SIZE = 8
LOG_EVERY = 500
SAVE_EVERY = 5000
MAX_AUDIO_SEC = 8.0
SAMPLE_RATE = 24000

print(f'[config] XLA available: {XLA_AVAILABLE}')
print(f'[config] model: {MODEL_NAME}')
print(f'[config] steps: {NUM_TRAIN_STEPS}, warmup: {WARMUP_STEPS}, lr: {LEARNING_RATE}')
print(f'[config] batch_size: {BATCH_SIZE}, bucket_sizes: {BUCKET_SIZES}')

In [ ]:
# ── Load secrets ────────────────────────────────────────────────────────────
secrets = load_secrets(require_gemini=False)
HF_TOKEN = secrets['HF_TOKEN_PRIMARY']
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# ── Read hf_repos.yaml ─────────────────────────────────────────────────────
with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO  = repos_cfg['repos']['stage0_codec']['repo_id']
STAGE45_REPO = repos_cfg['repos']['stage45_e2e']['repo_id']
HF_API = HfApi(token=HF_TOKEN)

print(f'[config] stage0_repo:  {STAGE0_REPO}')
print(f'[config] stage45_repo: {STAGE45_REPO}')

# ── Initialize WorkflowKernel ───────────────────────────────────────────────
run_id      = f'p5-finetune-{datetime.utcnow().strftime("%Y%m%d-%H%M%S")}'
session_id  = os.environ.get('KAGGLE_SESSION_ID', run_id)
kernel = WorkflowKernel(
    run_id=run_id,
    session_id=session_id,
    session_type='p5_finetune',
    shard_key='default',
    cf_worker_url=secrets['CF_WORKER_URL'],
    cf_worker_secret=secrets['CF_WORKER_SECRET'],
    gpu_type='tpu-v3-8',
    vram_limit_gb=16.0,
    session_max_hours=8.5,
)
kernel.start()
stage_start_ts = kernel.log_stage_start('p5_finetune')

# ── Download filtered dataset from stage0 ───────────────────────────────────
# Load the metadata parquet and filter segments where sample_for_tts_train=True
print('[dataset] loading stage0 metadata with TTS train filter...')
from datasets import load_dataset as hf_load_dataset

# Download the metadata which contains the sample_for_tts_train flag
metadata_ds = hf_load_dataset(
    STAGE0_REPO,
    data_dir='metadata',
    split='train',
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Filter for TTS training samples
tts_segments = metadata_ds.filter(lambda x: x.get('sample_for_tts_train', False))
print(f'[dataset] total segments with sample_for_tts_train=True: {len(tts_segments)}')

# Calculate total audio hours
if 'duration' in tts_segments.column_names:
    total_hours = sum(tts_segments['duration']) / 3600.0
    print(f'[dataset] total TTS audio: {total_hours:.1f} hours')
    assert total_hours >= 10, f'Expected >=10h of TTS audio, got {total_hours:.1f}h'
else:
    print('[dataset] duration column not found — proceeding with available segments')

# Load audio dataset with resampling to 24kHz
tts_ds = tts_segments.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
print(f'[dataset] ready — {len(tts_ds)} segments for TTS fine-tuning')

In [ ]:
# ── Load MMS-TTS model and processor ───────────────────────────────────────
from transformers import VitsModel, VitsTokenizer, AutoProcessor

print(f'[model] loading {MODEL_NAME}...')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME)
print(f'[model] loaded — params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

# ── Setup Accelerator / XLA device ─────────────────────────────────────────
if XLA_AVAILABLE:
    device = xm.xla_device()
    print(f'[device] XLA device: {device}')
    accelerator = Accelerator(device_placement=True, mixed_precision='no')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[device] fallback: {device}')
    accelerator = Accelerator()

model = model.to(device)

# ── Data collator with XLA shape bucketing ─────────────────────────────────
class BucketedCollator:
    """Pads batch to the nearest XLA bucket size for stable compilation."""

    def __init__(self, processor, bucket_sizes=None, max_audio_sec=8.0, sample_rate=24000):
        self.processor = processor
        self.bucket_sizes = bucket_sizes or BUCKET_SIZES
        self.max_samples = int(max_audio_sec * sample_rate)
        self.sample_rate = sample_rate

    def __call__(self, batch):
        texts = []
        waveforms = []
        for item in batch:
            text = item.get('transcript', item.get('text', ''))
            texts.append(text)
            audio = item['audio']['array']
            if isinstance(audio, np.ndarray):
                audio = torch.FloatTensor(audio)
            # Trim to max length
            if len(audio) > self.max_samples:
                audio = audio[:self.max_samples]
            waveforms.append(audio)

        # Find the max waveform length in this batch
        max_len = max(len(w) for w in waveforms)
        # Quantize to the nearest bucket for XLA shape stability
        bucket_len = quantize_to_bucket(max_len)
        bucket_len = min(bucket_len, self.max_samples)

        # Pad all waveforms to bucket_len
        padded_waveforms = []
        for w in waveforms:
            pad_len = bucket_len - len(w)
            if pad_len > 0:
                w = torch.nn.functional.pad(w, (0, pad_len))
            padded_waveforms.append(w)

        waveform_batch = torch.stack(padded_waveforms)

        # Process text with the VITS processor/tokenizer
        input_ids_list = [self.processor.tokenizer(t, return_tensors='pt').input_ids.squeeze(0) for t in texts]
        max_text_len = max(t.size(0) for t in input_ids_list)
        padded_input_ids = []
        attention_masks = []
        for ids in input_ids_list:
            pad_len = max_text_len - ids.size(0)
            padded_input_ids.append(torch.nn.functional.pad(ids, (0, pad_len), value=0))
            mask = torch.ones(ids.size(0))
            mask = torch.nn.functional.pad(mask, (0, pad_len), value=0)
            attention_masks.append(mask)

        input_ids = torch.stack(padded_input_ids)
        attention_mask = torch.stack(attention_masks)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'waveform': waveform_batch,
            'bucket_len': bucket_len,
            'waveform_lengths': torch.LongTensor([len(w) for w in waveforms]),
        }

collator = BucketedCollator(processor, max_audio_sec=MAX_AUDIO_SEC, sample_rate=SAMPLE_RATE)

# ── Create DataLoader ───────────────────────────────────────────────────────
from torch.utils.data import DataLoader

dataloader = DataLoader(
    tts_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=not XLA_AVAILABLE,
    drop_last=True,
)
print(f'[dataloader] {len(dataloader)} batches, batch_size={BATCH_SIZE}')

# ── Setup optimizer (AdamW) ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

# ── LR scheduler with warmup ────────────────────────────────────────────────
from torch.optim.lr_scheduler import LambdaLR

def warmup_cosine_schedule(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, NUM_TRAIN_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=warmup_cosine_schedule)

# Prepare with Accelerator
model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)
print(f'[training] optimizer and scheduler ready — {NUM_TRAIN_STEPS} steps, {WARMUP_STEPS} warmup')

In [ ]:
# ── Training Loop ───────────────────────────────────────────────────────────
model.train()
global_step = 0
data_iter = iter(dataloader)
train_start = time.time()
running_loss = 0.0
loss_count = 0
last_save_step = 0

print(f'[train] starting training for {NUM_TRAIN_STEPS} steps...')

while global_step < NUM_TRAIN_STEPS:
    # Check session expiry
    try:
        kernel.check_session_time()
    except Exception as e:
        print(f'[kernel] session expiring — saving checkpoint at step {global_step}')
        break

    # Get next batch (cycle dataloader)
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        batch = next(data_iter)

    input_ids      = batch['input_ids'].to(device)
    attention_mask  = batch['attention_mask'].to(device)
    waveform       = batch['waveform'].to(device)
    waveform_lengths = batch['waveform_lengths'].to(device)
    bucket_len     = batch['bucket_len']

    optimizer.zero_grad()

    # Forward pass — VITS model computes mel + generator + discriminator losses
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=waveform.unsqueeze(1),  # (B, 1, T)
    )

    loss = outputs.loss
    if loss is None:
        # Fallback: compute L1 spectrogram loss manually if labels not accepted
        with torch.no_grad():
            mel_target = model.get_encoder()(waveform.unsqueeze(1))
        mel_pred = outputs.spectrogram
        loss = torch.nn.functional.l1_loss(mel_pred, mel_target)

    accelerator.backward(loss)

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

    optimizer.step()
    scheduler.step()

    # XLA mark step for graph compilation
    if XLA_AVAILABLE:
        xm.mark_step()

    global_step += 1
    running_loss += loss.item()
    loss_count += 1

    # ── Logging ─────────────────────────────────────────────────────────────
    if global_step % LOG_EVERY == 0:
        avg_loss = running_loss / loss_count
        elapsed = time.time() - train_start
        steps_per_sec = global_step / max(elapsed, 1)
        eta_sec = (NUM_TRAIN_STEPS - global_step) / max(steps_per_sec, 1e-6)
        lr = scheduler.get_last_lr()[0]

        print(
            f'[train] step {global_step}/{NUM_TRAIN_STEPS} | '
            f'loss={avg_loss:.4f} | lr={lr:.2e} | '
            f'speed={steps_per_sec:.2f} steps/s | '
            f'ETA={eta_sec/3600:.1f}h | bucket={bucket_len}'
        )

        running_loss = 0.0
        loss_count = 0

    # ── Save checkpoint ─────────────────────────────────────────────────────
    if global_step % SAVE_EVERY == 0 and global_step > last_save_step:
        last_save_step = global_step
        ckpt_path = CHECKPOINT_DIR / f'step_{global_step}'
        ckpt_path.mkdir(parents=True, exist_ok=True)

        unwrapped = accelerator.unwrap_model(model)
        unwrapped.save_pretrained(str(ckpt_path))
        processor.save_pretrained(str(ckpt_path))

        # Save training state
        train_state = {
            'global_step': global_step,
            'loss': running_loss / max(loss_count, 1),
            'learning_rate': scheduler.get_last_lr()[0],
            'timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        }
        with open(ckpt_path / 'training_state.json', 'w') as f:
            json.dump(train_state, f, indent=2)

        print(f'[checkpoint] saved at step {global_step} → {ckpt_path}')

        # Flush memory
        kernel.flush_memory()

# ── Final save ──────────────────────────────────────────────────────────────
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
final_ckpt = CHECKPOINT_DIR / 'final'
final_ckpt.mkdir(parents=True, exist_ok=True)

unwrapped = accelerator.unwrap_model(model)
unwrapped.save_pretrained(str(final_ckpt))
processor.save_pretrained(str(final_ckpt))

final_state = {
    'global_step': global_step,
    'total_steps': NUM_TRAIN_STEPS,
    'elapsed_hours': (time.time() - train_start) / 3600.0,
    'timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
}
with open(final_ckpt / 'training_state.json', 'w') as f:
    json.dump(final_state, f, indent=2)

print(f'\n[train] training complete — {global_step} steps in {(time.time()-train_start)/3600:.1f}h')
print(f'[train] final checkpoint: {final_ckpt}')

In [ ]:
# ── Upload checkpoint to stage45_e2e/checkpoints/ ───────────────────────────
print('[upload] uploading MMS-TTS checkpoint to HF...')

final_ckpt = CHECKPOINT_DIR / 'final'
ckpt_files = list(final_ckpt.rglob('*'))
ckpt_files = [f for f in ckpt_files if f.is_file()]
print(f'[upload] {len(ckpt_files)} files to upload')

for fpath in ckpt_files:
    rel = fpath.relative_to(final_ckpt)
    repo_path = f'checkpoints/mms_tts_finetuned/{rel}'
    for attempt in range(8):
        try:
            HF_API.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=repo_path,
                repo_id=STAGE45_REPO,
                repo_type='dataset',
                commit_message=f'p5: upload MMS-TTS checkpoint ({rel.name})',
            )
            break
        except Exception as e:
            wait = min(2 ** attempt, 120)
            print(f'  [retry] {rel} attempt {attempt+1}/8: {e} — retry in {wait}s')
            time.sleep(wait)

# Upload intermediate checkpoints too
for step_dir in sorted(CHECKPOINT_DIR.glob('step_*')):
    step_files = [f for f in step_dir.rglob('*') if f.is_file()]
    step_name = step_dir.name
    print(f'[upload] uploading intermediate checkpoint {step_name} ({len(step_files)} files)...')
    for fpath in step_files:
        rel = fpath.relative_to(step_dir)
        repo_path = f'checkpoints/mms_tts_finetuned/{step_name}/{rel}'
        for attempt in range(6):
            try:
                HF_API.upload_file(
                    path_or_fileobj=str(fpath),
                    path_in_repo=repo_path,
                    repo_id=STAGE45_REPO,
                    repo_type='dataset',
                    commit_message=f'p5: upload {step_name} checkpoint',
                )
                break
            except Exception:
                time.sleep(min(2 ** attempt, 60))

print(f'[upload] all checkpoints uploaded to {STAGE45_REPO}/checkpoints/mms_tts_finetuned/')

# ── Log completion to CF Worker ─────────────────────────────────────────────
kernel.log_stage_end('p5_finetune', stage_start_ts)
kernel.stop()

print('\n' + '='*60)
print('  p5_finetune COMPLETE')
print(f'  Model: facebook/mms-tts-urd-script_arabic (fine-tuned)')
print(f'  Checkpoint: {STAGE45_REPO}/checkpoints/mms_tts_finetuned/')
print(f'  Steps trained: {global_step}')
print('='*60)